In [2]:
import time 
import pygame

pygame 2.6.1 (SDL 2.28.4, Python 3.13.14)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
class Objet:
    def __init__(self, x, y,):
        self.x = x
        self.y = y 


class Joueur(Objet):
    def __init__(self, x, y, vy, vx):
        super().__init__(x,y)
        self.vy = vy
        self.vx = vx
        self.gravite = -0.9

class carre(Objet):
    def __init__(self, x, y, taille):
        super().__init__(x,y)
        self.taille = taille

class plateforme(Objet):
    def __init__(self, x, y):
        super().__init__(x,y)
    

class pique(Objet):
    def __init__(self, x, y, taille):
        super().__init__(x,y)
        self.taille = taille

In [ ]:
class Jeu:
    def __init__(self):
        self.joueur = Joueur(0, 0, 0, 6)
        self.obstacles = []
        self.score = 0

    def ajouter_obstacle(self, obstacle):
        self.obstacles.append(obstacle)

    def saut(self):
        if self.joueur.y == 0 or self.surplateform(self.joueur.x, self.joueur.y):
            self.joueur.vy = 15
        

    def deplacer_obstacles(self):
        for obstacle in self.obstacles:
            obstacle.x -= self.joueur.vx

    def surplateform(self, x, y):
        for obstacle in self.obstacles:
            if isinstance(obstacle, plateforme):
                chevauchement = (
                    x + 50 > obstacle.x
                    and x < obstacle.x + 150
                )
                if chevauchement and abs(y - obstacle.y) < 1:
                    return True
        return False

    def plateforme_traversee(self, y_avant, y_apres):
        for obstacle in self.obstacles:
            if not isinstance(obstacle, plateforme):
                continue
            chevauchement = (
                self.joueur.x + 50 > obstacle.x
                and self.joueur.x < obstacle.x + 150
            )
            if chevauchement and y_avant >= obstacle.y >= y_apres:
                return obstacle
        return None
        
    def mettre_a_jour_joueur(self):
        y_avant = self.joueur.y
        self.joueur.vy += self.joueur.gravite
        y_apres = y_avant + self.joueur.vy

        plateforme_touchee = None
        if self.joueur.vy <= 0:
            plateforme_touchee = self.plateforme_traversee(y_avant, y_apres)

        if plateforme_touchee is not None:
            self.joueur.y = plateforme_touchee.y
            self.joueur.vy = 0
        elif y_apres <= 0:
            self.joueur.y = 0
            self.joueur.vy = 0
        else:
            self.joueur.y = y_apres

    def mettre_a_jour_score(self):
        self.score += 1

        if self.score % 10 == 0:
            self.joueur.vx += 0.01

    def verifier_collision(self):
        for obstacle in self.obstacles:

            if isinstance(obstacle, plateforme):
                
                continue  

            else:
                if isinstance(obstacle, carre) and (
                    
                    abs(obstacle.x - self.joueur.x)
                    < min(50, obstacle.taille)
                    and
                    abs(obstacle.y - self.joueur.y)
                    < min(50, obstacle.taille)
                ):
                    return True
                if isinstance(obstacle, pique):
                    gauche = max(self.joueur.x, obstacle.x)
                    droite = min(self.joueur.x + 50, obstacle.x + obstacle.taille)

                    if gauche > droite:
                        continue
                    sommet_x = obstacle.x + obstacle.taille / 2
                    X = min(max(sommet_x, gauche), droite)

                    # Hauteur du pique en X
                    hauteur_pique = (
                        obstacle.y
                        + obstacle.taille
                        - 2 * abs(X - sommet_x)
                                )

                    if (
                        self.joueur.y <= hauteur_pique
                    and self.joueur.y + 50 >= obstacle.y
                    ):
                        return True
                    else:
                        continue

                

        return False

    def jouer(self):
        affichage = Affichage(self)
        running = True

        while running:
            self.deplacer_obstacles()
            self.mettre_a_jour_joueur()
            self.mettre_a_jour_score()

            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_SPACE:
                        self.saut()

            if self.verifier_collision():
                print("Collision! Game Over.")
                running = False

            affichage.dessiner()

            pygame.display.flip()
            time.sleep(0.016)  # 60 FPS
        pygame.quit()    

In [5]:
class Affichage:
    def __init__(self, jeu):
        self.jeu = jeu

        pygame.init()
        self.ecran = pygame.display.set_mode((800, 600))
        pygame.display.set_caption("GD Test")


    def dessiner_obstacle(self, obstacle):

        if isinstance(obstacle, carre):
            pygame.draw.rect(
                self.ecran,
                pygame.Color("red"),
                pygame.Rect(
                    obstacle.x,
                    600 - obstacle.y - obstacle.taille,
                    obstacle.taille,
                    obstacle.taille
                )
            )
            

        elif isinstance(obstacle, plateforme):
            largeur = 150
            hauteur = 20

            x = obstacle.x
            y = 600 - obstacle.y

            pygame.draw.rect(
                self.ecran,
                pygame.Color("green"),
                pygame.Rect(
                    x,
                    y - hauteur + 20,
                    largeur,
                    hauteur
                )
            )

        elif isinstance(obstacle, pique):
            x = obstacle.x
            y = 600 - obstacle.y

            pygame.draw.polygon(
                self.ecran,
                pygame.Color("orange"),
                [
                    (x, y),
                    (x + obstacle.taille, y),
                    (x + obstacle.taille / 2, y - obstacle.taille)
                ]
            )

    def dessiner(self):
        self.ecran.fill(pygame.Color("black"))
        pygame.draw.rect(
            self.ecran,
            pygame.Color("white"),
            pygame.Rect(
                self.jeu.joueur.x,
                600 - self.jeu.joueur.y - 50,
                50,
                50
            )
        )

        for obstacle in self.jeu.obstacles:
            self.dessiner_obstacle(obstacle)

In [26]:
jeu = Jeu()

def niveau1(jeu):

    # Début tranquille
    jeu.ajouter_obstacle(pique(500, 0, 40))
    jeu.ajouter_obstacle(pique(750, 0, 40))

    # Petit enchaînement
    jeu.ajouter_obstacle(pique(950, 0, 40))
    jeu.ajouter_obstacle(pique(1000, 0, 40))

    # Première plateforme
    jeu.ajouter_obstacle(plateforme(1250, 100))
    jeu.ajouter_obstacle(pique(1250, 0, 50))
    jeu.ajouter_obstacle(pique(1300, 0, 50))
    jeu.ajouter_obstacle(pique(1350, 0, 50))

    # Obstacle après la plateforme
    jeu.ajouter_obstacle(pique(1750, 0, 40))

    # Partie un peu plus verticale
    jeu.ajouter_obstacle(plateforme(1950, 80))
    jeu.ajouter_obstacle(plateforme(2200, 150))

    # Retour au sol
    jeu.ajouter_obstacle(pique(2500, 0, 40))
    jeu.ajouter_obstacle(pique(2550, 0, 40))

    # Petit final
    jeu.ajouter_obstacle(pique(2800, 0, 50))
    jeu.ajouter_obstacle(pique(3000, 0, 40))
    jeu.ajouter_obstacle(pique(3300, 0, 40))

    jeu.jouer()

def niveautest(jeu):
    for i in range(30):
        jeu.ajouter_obstacle(plateforme(500*(i+1), 40))
    jeu.jouer()

niveau1(jeu)

#A faire (en quasi autonomie):
#2) gérer les plateformes
#3) gérer le score

Collision! Game Over.
